# Kimodo → Mixamo retargeter — Kaggle, CPU only (web UI + 3D viewport)

The full Kimodo demo is too heavy for free-tier GPU sessions — but the **retargeter
is pure numpy** and runs fine on a plain CPU. This notebook:

1. clones your extended fork,
2. installs a few tiny packages (fastapi, uvicorn, pygltflib, numpy — **no torch**),
3. starts `retarget/webapp.py` — a web app where you upload
   - your **character GLB** (Mixamo-rigged), and
   - a **motion file**: `.npz` from `kimodo_gen`, or `.bvh` from the Kimodo demo export,
4. prints a **public trycloudflare.com URL** — open it, the browser shows the retargeted
   animation in a 3D viewport, and **Download GLB** gives you the baked result.

**Pick the plain CPU accelerator** (no GPU quota needed). Runtime cost: ~zero.

In [ ]:
#@title 0. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/liladhiee-web/kimodo-extended.git"  #@param ["https://github.com/liladhiee-web/kimodo-extended.git", "https://github.com/nv-tlabs/kimodo.git"]
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/kimodo"
SERVER_PORT = 8000
print("config ok ->", REPO_URL)

## 2 · Clone the repo

Defaults to **your extended fork** (`retarget/` tooling included). Pulls if already present.

In [ ]:
import os, shutil, subprocess as sp
if os.path.isdir(f"{REPO_DIR}/.git"):
    r = sp.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], capture_output=True, text=True)
    print("repo present, pull:", (r.stdout or r.stderr).strip() or "up-to-date")
else:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    os.makedirs(WORK, exist_ok=True)
    r = sp.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git clone failed: " + r.stderr[-2000:])
    print("cloned into", REPO_DIR)
os.chdir(REPO_DIR)
r2 = sp.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
print("git HEAD:", r2.stdout.strip())

## 3 · Install dependencies (~1 min)

numpy is preinstalled on Kaggle; the rest are tiny pure-Python wheels. No torch anywhere.

In [ ]:
import subprocess, sys, os

def pip(*args, label):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    print("pip " + label + ": " + ("ok" if r.returncode == 0 else "FAILED\n" + r.stderr[-1500:]))

pip("pygltflib", "fastapi", "uvicorn", "python-multipart", label="web + retarget deps")

sys.path.insert(0, os.path.join(REPO_DIR, "retarget"))
import numpy, pygltflib, fastapi, uvicorn
import glb, mapping, quatmath, retarget, soma_bvh, webapp  # retarget package import check
print(f"numpy {numpy.__version__} | pygltflib {pygltflib.__version__} | fastapi {fastapi.__version__}")
print("retarget modules import: ok — CPU only, no torch required")

## 4 · Launch the web app

In [ ]:
import os, subprocess, sys, time, urllib.request

subprocess.run(["pkill", "-f", "webapp.py"], capture_output=True)
time.sleep(1)

os.environ["PORT"] = str(SERVER_PORT)
os.environ["HOST"] = "0.0.0.0"
app_log = open(f"{WORK}/webapp.log", "w")
proc = subprocess.Popen(
    [sys.executable, "webapp.py"],
    cwd=os.path.join(REPO_DIR, "retarget"),
    stdout=app_log,
    stderr=subprocess.STDOUT,
)
print("webapp PID:", proc.pid)

ok = False
for _ in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen(f"http://localhost:{SERVER_PORT}/api/health", timeout=3)
        ok = True
        break
    except Exception:
        pass

print("=" * 72)
print("  webapp:", "UP on port", SERVER_PORT if ok else "NOT UP — peek at /kaggle/working/webapp.log")
print("=" * 72)

## 5 · Cloudflare quick-tunnel → your public URL

In [ ]:
import os, re, subprocess, time

CF_BIN = f"{WORK}/cloudflared"
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
if not os.path.exists(CF_BIN):
    subprocess.run(
        [
            "wget", "-q", "-O", CF_BIN,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
subprocess.run(["chmod", "+x", CF_BIN])

tunnel_log = open(f"{WORK}/tunnel.log", 'wb')
tunnel_proc = subprocess.Popen(
    [CF_BIN, 'tunnel', '--no-autoupdate', f'--url=http://localhost:{SERVER_PORT}'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)
print("tunnel PID:", tunnel_proc.pid)

public_url = None
for _ in range(40):
    time.sleep(2)
    try:
        text = open(f"{WORK}/tunnel.log", 'rb').read().decode('utf-8', errors='replace')
    except OSError:
        continue
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", text)
    if m:
        public_url = m.group(0)
        break

print("=" * 72)
print("  YOUR PUBLIC RETARGETER:")
print(f"      {public_url}" if public_url else "      (not ready yet — rerun this cell)")
print("=" * 72)
print("open the URL, upload your character GLB + a motion file (.npz / .bvh), press Retarget.")

## 6 · Keep-alive

In [ ]:
import time, urllib.request
try:
    beat = 0
    while True:
        time.sleep(60)
        beat += 1
        state = "webapp: up"
        try:
            urllib.request.urlopen(f"http://localhost:{SERVER_PORT}/api/health", timeout=5)
        except Exception:
            state = "webapp: DOWN (rerun cells 4 & 5)"
        print(f"[{beat:>4}] {state}")
except KeyboardInterrupt:
    print("keepalive stopped — rerun cells 4-6 to come back")

## 7 · Where do the input files come from?

| Input | How to get it |
|---|---|
| **Character `.glb`** | Your Mixamo-rigged character exported as **glTF Binary (.glb)**, T-pose. A Blender glTF export works too (needs `mixamorig:` bone names). |
| **Motion `.npz`** | Run `kimodo_gen` locally on a machine where Kimodo works (GPU), or the full-demo notebook `kaggle_cloudflared_kimodo.ipynb` if you get GPU time. The CLI output NPZ is already the right SOMA-77 format. |
| **Motion `.bvh`** | Kimodo **web demo** → download → BVH. If the export offers a "standard T-pose" option, leave it **on** — it matches the rest pose our limb-length fix assumes. |

Notes:
- `.npz` inputs carry **foot contacts**, so *sticky feet* IK works; `.bvh` inputs have
  none — the app auto-disables sticky feet and tells you so.
- The mapping report you get right after the character upload lists exactly which of
  the 77 joints mapped to your rig — a character missing fingers still retargets, minus those bones.

## Troubleshooting

| Symptom | Fix |
|---|---|
| Tunnel shows 502 | webapp not running — rerun **cell 4** (launch) then **cell 5** (tunnel) |
| Upload rejected *could not parse GLB* | character must be `.glb`; if your rig is FBX, re-export as glTF Binary |
| Upload rejected *could not parse motion* | motion must be a Kimodo `.npz` (SOMA-77) or `.bvh`; other mocap NPZ keys won't work |
| Few mapped bones in report | the character isn't Mixamo-named; check the report listing after upload |
| Webapp dead after idle | rerun cell 4 & 5; keepalive cell prevents idle timeouts |
| Job error in log box | read the log lines in the UI; `webapp.log` has the full traceback |